# How to contruct dot array system  

## 0. Import relevant packages and QuDiPy modules

In [ ]:
import os, sys
sys.path.append(os.path.dirname(os.getcwd()))

import numpy as np
import matplotlib.pyplot as plt

from qudipy.dot_array import DotArray

## 1. Generate Dots object instance


### 1.1 Define variables to initialize Dots object instance

Define the directory to  <span style="background-color: Khaki;">store 2D potential slices</span> for all the nextnano simulation data for a given device.

In [ ]:
preprocessed_dir = os.path.join(sys.path[0], 'QuDiPy tutorial data','Pre-processed potentials','Pre-processed_data')

Define the directory location that contains all of the nextnano simulation  <span style="background-color: Khaki;">data files to be processed or imported</span>.

In [ ]:
data_dir = os.path.join(os.getcwd(), 'QuDiPy tutorial data','Nextnano simulations','2QD_dotsep_60nm')

Define a directory to  <span style="background-color: Khaki;">store calculated variables</span> such as $g$-factor deviation, exchange, or potentials for individual dots or dot pairs.

In [ ]:
precalculated_dir = os.path.join(os.getcwd(), 'QuDiPy tutorial data','Pre-processed potentials')

Define a list of control names for the contacts of the device simulated in nextnano. 
> **Note**: make sure to label these the same as they are defined in the nextnano input script.

In [ ]:
ctrl_names = ['GateContact_T1','GateContact_P1','GateContact_T2',
                    'GateContact_P2','GateContact_T3','GateContact_S',
                            'SiContact']
n_dots = 2      # anticipated number of dots

Define control parameters to use when interpolating the potential data from nextnano: must be a subset of the raw data. 
> **Note**: These will greatly influence the accuracy of the resultant calculations as well as the run time of the `Dots()` class upon initial data processing.

In [ ]:
pts = [3,3,3]
ctrl_vals = [[0.0], 
    np.linspace(0.2, 0.4, pts[0]),
    np.linspace(-0.1, 0.1, pts[1]),
    np.linspace(0.2, 0.4, pts[2]),
    [0.0], [0.0], [0.0]]

Add a file prefix name to be appended to saved data or interpolation objects that will be computed.

In [ ]:
file_prefix = 'data'

### 1.2 Create object instance 

First, instantiate a `Dots()` class object instance using the previously defined variables and choose to calculate the effective parameters ($g$-factor deviation and exchange), or choose just to collect the masked potential interpolation objects (currently these values must be set both to `True`).

If no nextnano data has been saved to the Pre-process directory, then the `save=True` argument must be included. 
This is  <span style="background-color: Khaki;">only needed for the first attempted execution</span>. After the nextnano data has been processed and saved to the Pre-process directory, this flag can be removed.


In [ ]:
        # UNCOMMENT FOR THE FIRST CALCULATION
    # Calculate the effective parameters and/or load the masked potential 
    # lanscapes as well as save 2D potential slices to Pre-processed directory
# dots = DotArray(precalculated_dir, data_dir, preprocessed_dir, n_dots,
#                     ctrl_names, ctrl_vals, file_prefix, calc_eff=True, pts=pts, save=True)

    # UNCOMMENT FOR ALL SUCCESSIVE CALCULATIONS
# Calculate the effective parameters and/or load the masked potential lanscapes
# dots = DotArray(precalculated_dir, data_dir, preprocessed_dir, n_dots,
#                     ctrl_names, ctrl_vals, file_prefix, calc_eff=True, pts=pts)

# Calculate just the masked potential lanscapes
dots = DotArray(precalculated_dir, data_dir, preprocessed_dir, n_dots,
                     ctrl_names, ctrl_vals, file_prefix, pts=pts)


## 1.3 Test dot splitting

The splitting procedure creates a list of new Dots objects. 
They inherit all properties from the parent object _except_:
1. potential interpolator
`dots_object.potential(ctrl_vals)` now outputs masked potential
2. the positions of visible dots __*(counted from 1)*__ may change.

Splitting group can be either `'single'`/`'singles'` (default) for individual dots, 
or `'pair'`/`'pairs'` for adjacent pairs.


> **Note**: if the number of quantum dots decreases for any control value coordinates, then the potential landscape returns `NaN`s, since the quantum dot system must be dot invariant for the present applications. 

> `NaN`s are not displayed when plotting

In [ ]:
# Choose voltage control values
voltage_config = [0.2, 0.1, 0.3]

# Quantum dots are not well defined in this case, so NaNs are outputted and 
# are not plotted
# voltage_config = [0.2,0.,0.4]

dot1, dot2 = dots.split(group='single')

print('Visible dots:\t', dot1.visible_dots)
dot1.potential.plot(voltage_config)

All Dots objects preserve information about the unmasked potential.
They can also be split more than once.

In [ ]:
dot1.potential_unmasked.plot(voltage_config)

# Try splitting dot1 again
# ddot1, = dot1.split()
# ddot1.potential.plot(voltage_config)

## 1.4 Evaluate effective parameters

For a specific control voltage configuration, the $g$-factor deviations and exchange couplings are determined for each dot or neighboring pair, respectively.

### 1.4.1 g-factor deviation

g-factor deviation for a single voltage vector

In [ ]:
# higher voltage on the 2nd dot gives higher g-factor deviation

voltage_config = [0.2, 0.1, 0.3]
dots.g_factors(voltage_config)

g-factor deviation for multiple voltage vectors

In [ ]:
voltage_configs = []
v = np.linspace(0.2,0.3,10)
for volt in v:
    voltage_configs.append([volt,0.1,0.2])

dots.g_factors(voltage_configs)

Plotting g-factor deviation for a path through voltage state space

In [ ]:
dots.plot(voltage_configs, param='gfactor')

### 1.4.2 Hund-Mulliken or Heitler-London Exchange

Exchange for a single voltage vector

In [ ]:
# HM values are typically larger than HL
ex_hl = dots.exchanges(voltage_config, method='hl')
ex_hm = dots.exchanges(voltage_config, method='hm')

# convert values to neV
print(f'HL exchange: \t {ex_hl[0] / 1.6e-28:.5} neV')
print(f'HM exchange: \t {ex_hm[0] /  1.6e-28:.5} neV')

Exchange for multiple voltage vectors

In [ ]:
voltage_configs = []
v = np.linspace(0.0,0.1,10)
for volt in v:
    voltage_configs.append([0.25, volt, 0.2])

dots.exchanges(voltage_configs, method='hl')
dots.exchanges(voltage_configs, method='hm')

Plotting exchange for a path through voltage state space

In [ ]:
dots.plot(voltage_configs, param='hm', ex_units='neV', yscale='linear')
dots.plot(voltage_configs, param='hl', ex_units='ueV', yscale='log')

### 1.5 Calculate effective parameter interpolation objects
Next, call the `get_eff_params()` method to perform the necessary calculations and save the results, or load previously calculated results.

In [ ]:
dots.get_eff_params()

## 2. Accessing attributes of a Dots object instance

### 2.1 Accessing potential interpolation objects

The `Dots()` class will determine the maximum number of quantum dots for a given data set and then finds the potential lanscaped for individual quantum dots or adjacent quantum dot pairs. The potential landscapes found in these circumstances is performed via a masking procedure. 


In [ ]:
# some test cases showing potential bugs

# unmasked and masked exchange should be the same
v = [0.2,0.1,0.28]

# other cases
# v = [0.45,0.1,0.45]
# v = [0.45,0.0,0.2]
# v = [0.2,0.1,0.2]

# convert to nm
units = 1e-9

# extract coordinates
x = dots.pot_interp_dict['unmasked'].x_coords / units
y = dots.pot_interp_dict['unmasked'].y_coords / units

# define extent to plot potential lanscapes with coordinates
dx = (x[1]-x[0])/2.
dy = (y[1]-y[0])/2.
extent = [x[0]-dx, x[-1]+dx, y[0]-dy, y[-1]+dy]

fig, ax = plt.subplots(2,2, figsize=(5,3), dpi=150)

pot = dots.pot_interp_dict['unmasked'](v)
ax[0,0].imshow(pot, extent=extent)

pot = dots.pot_interp_dict['masked_J_12'](v)
ax[0,1].imshow(pot, extent=extent)

pot = dots.pot_interp_dict['masked_delta_g1'](v)
ax[1,0].imshow(pot, extent=extent)

pot = dots.pot_interp_dict['masked_delta_g2'](v)
ax[1,1].imshow(pot, extent=extent)

ax[0,0].set_ylabel('y [nm]')
ax[1,0].set_ylabel('y [nm]')

ax[1,0].set_xlabel('x [nm]')
ax[1,1].set_xlabel('x [nm]')

fig.tight_layout()

plt.show()

### 2.1 Accessing effective parameter interpolation objects

A simple example exctracting the effective parameter interpolation objects and plotting verse control values. 

> **Note** NaNs are not displayed when plotting

In [ ]:

v = np.linspace(0.205,0.35, 100)
t = np.linspace(-0.1,0.1, 100)

vr = [[0.205], [0.0], v]
vl = [v, [0.0], [0.205]]
vt = [[0.205], t, [0.205]]

J_t = dots.eff_interp_dict['J_12'](*vt)

fig, ax = plt.subplots(1,3, figsize=(8,3), dpi=150)

# bug in exchange interpolation object
ax[0].plot(v,np.squeeze(dots.eff_interp_dict['J_12'](*vr)), '-', label='J($V_R$)')
ax[0].plot(v,np.squeeze(dots.eff_interp_dict['J_12'](*vl)), '-', label='J($V_L$)')
ax[0].plot(t,np.squeeze(dots.eff_interp_dict['J_12'](*vt)), '-', label='J($V_T$)')
ax[0].set_ylabel('Exchange (J)')
ax[0].legend()


ax[1].plot(v,np.squeeze(dots.eff_interp_dict['delta_g1'](*vr)), '-', label='g($V_R$)')
ax[1].plot(v,np.squeeze(dots.eff_interp_dict['delta_g1'](*vl)), '-', label='g($V_L$)')
ax[1].plot(t,np.squeeze(dots.eff_interp_dict['delta_g1'](*vt)), '-', label='g($V_T$)')
ax[1].set_xlabel('Voltage')
ax[1].set_ylabel('$\delta$g Left Dot')
ax[1].legend()

ax[2].plot(v,np.squeeze(dots.eff_interp_dict['delta_g2'](*vr)), '-', label='J($V_R$)')
ax[2].plot(v,np.squeeze(dots.eff_interp_dict['delta_g2'](*vl)), '-', label='J($V_L$)')
ax[2].plot(t,np.squeeze(dots.eff_interp_dict['delta_g2'](*vt)), '-', label='J($V_T$)')
ax[2].set_ylabel('$\delta$g Right Dot')
ax[2].legend()
fig.tight_layout()